# Prepare Cone Calorimeter Experimental Data

.




In [ ]:
import os
import re
import sys
import fdsreader
import matplotlib

import numpy as np
import scipy as sp
import pandas as pd
import firescipy as fsp
import utilities as utils

import matplotlib.pyplot as plt

from importlib import reload  # Python 3.4+
from scipy.ndimage import uniform_filter1d
from scipy.signal import savgol_filter

# fsp_dev_path = os.path.join("..", "..", "..", "Kinetics", "ReactionRateDev", "GeneralInformation")
# # Add the directory containing your module to the Python path (wants absolute paths)
# sys.path.append(os.path.abspath(fsp_dev_path))
# print(os.path.abspath(fsp_dev_path))
# # sys.path.append(fsp_dev_path)
# import FireSciPy as fsp
# reload(fsp)

reload(utils)


In [ ]:
###########################################################################
## ! Use the 'requirements.txt' to create a virtual Python environment ! ##
###########################################################################

# Package Versions
# ----------------
# Python version: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
# Python path: F:\PhD\BESKID\PyrolysisModelling\GeneralInformation\venv_BESKID_Pyro\Scripts\python.exe
# Numpy version: 2.3.2
# SciPy version: 1.16.1
# Pandas version: 2.3.1
# FireSciPy version: 0.0.5  # py -m pip install --index-url https://test.pypi.org/simple/ --no-deps firescipy==0.0.5
# Matplotlib version: 3.10.5
# RegEx version: 2.2.1


print('Package Versions')
print('----------------')
print('Python version: {}'.format(sys.version))
print('Python path: {}'.format(sys.executable))
print('Numpy version: {}'.format(np.__version__))
print('SciPy version: {}'.format(sp.__version__))
print('Pandas version: {}'.format(pd.__version__))
print('fdsreader version: {}'.format(fdsreader.__version__))
print('FireSciPy version: {}'.format(fsp.__version__))
print('Matplotlib version: {}'.format(matplotlib.__version__))
print('RegEx version: {}'.format(re.__version__))


In [ ]:
# General information.

# Root to experimental data
cone_root = os.path.join("..", "Experiments", "BESKID_ExperimentalData", 
                         "Materials", "PMMA", "DE_6mm","Cone")


# Order of plot colors
plot_colors = ["tab:blue", "tab:orange", "tab:green", 
               "tab:red", "tab:purple", "tab:brown", 
               "tab:pink", "tab:gray", "tab:olive", 
               "tab:cyan"]

# Basic plot settings
plt.rcParams.update({
    'axes.axisbelow': True,     # Keep grid behind plots
    'figure.autolayout': True,  # Equivalent to calling tight_layout()
    'axes.facecolor': 'white',  # Prevents transparent background
    'grid.alpha': 0.6,          # Makes gridlines more readable
    'font.size': 12             # Set global font size
})


# Directory used to collect the output produced by this notebook.
output_dir = "PrepareConeExperiments_Output"
if not os.path.isdir(output_dir):
    os.mkdir(output_dir)
    print("* Output directory created.")
else:
    print("* Output directory already exists.")


In [ ]:

def combine_data(repetitions, x_header, y_header):

    # Step 1: Find reference time from longest series
    longest_x = None
    for rep in repetitions.values():
        if longest_x is None or rep[x_header].iloc[-1] > longest_x.iloc[-1]:
            longest_x = rep[x_header]
    reference_x = longest_x.values

    # Step 2: Interpolate all repetitions to reference time
    combined_data = {x_header: reference_x}
    for rep_name, rep_data in repetitions.items():
        for col in [(y_header)]:
            if col not in rep_data.columns:
                raise ValueError(f"Missing column '{col}' in repetition '{rep_name}'.")

            combined_data[f"{y_header}_{rep_name}"] = np.interp(
                reference_x, rep_data[x_header], rep_data[col])

    # Step 3: Compute mean and std dev
    for col_default in [y_header]:
        values = [combined_data[k] for k in combined_data if k.startswith(f"{y_header}_")]
        combined_data[f"{y_header}_Avg"] = np.mean(values, axis=0)
        combined_data[f"{y_header}_Std"] = np.std(values, axis=0)

    # Step 4: Store data to file
    combined_df = pd.DataFrame(combined_data)
    
    return combined_df



# Read Experimental Data



In [ ]:

cone_records = dict()

for file_label in os.listdir(cone_root):
    if "README.md" in file_label:
        continue
    print(file_label)
    
    label_parts = file_label[:-4].split('_')
    q_label = label_parts[4]
    rep_label = label_parts[-1]
    
    file_path = os.path.join(cone_root, file_label)
    cone_meta, cone_data = utils.read_cone_data(file_path)
    dataset = {
        "meta": cone_meta,
        "data": cone_data}
    
    
    keys = [q_label, rep_label]
    fsp.utils.store_in_nested_dict(cone_records, dataset, keys)
    


In [ ]:

q_label = 'q25'

q_data_combined = dict()

sample_masses = dict()

seen = set()

for q_id, q_label in enumerate(cone_records):
    
    q_data = dict()
    q_masses = list()
    for rep_label in cone_records[q_label]:

        cone_rep_data = cone_records[q_label][rep_label]['data']
        cone_rep_meta = cone_records[q_label][rep_label]['meta']
        # Collect cone data
        q_data[rep_label] = cone_rep_data
        
        init_mass = float(cone_rep_meta['Initial mass (g)'].replace(',','.'))
        q_masses.append(init_mass)
        
        # Show raw data
        if q_label in seen:
            plot_label = None
        else:
            plot_label = f"q = {q_label[1:]} kW/m²"
            
        seen.add(q_label)
        
        plt.plot(cone_rep_data['time (s)'], 
                 cone_rep_data['HRR'], 
                 color=plot_colors[q_id], alpha=0.5,
                 label=plot_label)
    
    # Combine repetitions 
    df = combine_data(q_data, 'time (s)', 'HRR')
    q_data_combined[q_label] = df
    
    # Collect masses
    sample_masses[q_label] = q_masses
    
    # Write combined data to disk
    file_label = f"Cone_{q_label}.csv"
    file_path = os.path.join(output_dir, file_label)
    df.to_csv(file_path, index=False)
    
    # Show averaged result
    print(df['time (s)'].iloc[-1])
    plt.plot(df['time (s)'],
             df['HRR_Avg'],
             color='black')


# Check file
df = pd.read_csv(file_path, header=0)

plt.plot(df['time (s)'],
         df['HRR_Avg'],
         color='red', 
         linestyle='--')
    
    
# Plot meta data
plt.xlabel("Time / s")
plt.ylabel("Heat Release Rate / kW")

plt.legend()
plt.grid()


In [ ]:

q_labels = ['q50']

start_frame = 60
end_frame = 420

seen = set()

for q_id, q_label in enumerate(q_labels):
    
    q_data = dict()
    for rep_label in cone_records[q_label]:

        cone_rep_data = cone_records[q_label][rep_label]['data']
        # Collect cone data
        q_data[rep_label] = cone_rep_data
        
        # Show raw data
        if q_label in seen:
            plot_label = None
        else:
            plot_label = f"q = {q_label[1:]} kW/m²"
            
        seen.add(q_label)
        
        plt.plot(cone_rep_data['time (s)'][start_frame:end_frame], 
                 cone_rep_data['HRR'][start_frame:end_frame], 
                 color=plot_colors[q_id], alpha=0.5,
                 label=plot_label)
    
    # Combine repetitions 
    df = combine_data(q_data, 'time (s)', 'HRR')
    
    
    # Show averaged result
    print(df['time (s)'].iloc[-1])
    plt.plot(df['time (s)'][start_frame:end_frame],
             df['HRR_Avg'][start_frame:end_frame],
             color='black')
    
    
# Plot meta data
plt.xlabel("Time / s")
plt.ylabel("Heat Release Rate / kW")

plt.legend()
plt.grid()

file_label = f"Cone_{q_label}kW.png"
file_path = os.path.join(output_dir, file_label)

plt.savefig(file_path, dpi=320)


In [ ]:

q_label = 'q25'


initial_hrr = {
    "q25": 150,
    "q50": 80,
    "q75": 60}


for q_id, q_label in enumerate(q_data_combined):
    df = q_data_combined[q_label]
    
    time = df['time (s)']
    hrr = df['HRR_Avg']
    
    # Show averaged result
    print(time.iloc[-1])
    plt.plot(time,
             hrr,
             color='black', alpha=0.3)
    
    # Show the initial period
    time_cutoff = initial_hrr[q_label]
    idx_initial = (np.abs(time - time_cutoff)).argmin()
    print(time[idx_initial])
    plt.plot(time[:idx_initial],
             hrr[:idx_initial],
             color='black')
    
    
# Plot meta data
plt.xlabel("Time / s")
plt.ylabel("Heat Release Rate / kW")

plt.legend()
plt.grid()



## Determine Effective Heat of Combustion


In [ ]:

# Prepare data collection
cone_thr_all = dict()
cone_eff_hoc_all = dict()


# Determine total heat release (THR)
for q_id, q_label in enumerate(q_data_combined):
    # Get heat release
    df = q_data_combined[q_label]
    
    time = df['time (s)']
    hrr = df['HRR_Avg']
    
    thr = np.trapezoid(hrr, time)
    cone_thr_all[q_label] = thr
    
    print(f"THR ({q_label}): {thr:.8} kJ")
    
    # Get sample masses
    init_masses_q = sample_masses[q_label]
    init_mass_avg = np.average(init_masses_q) / 1000
    
    print(f"Sample mass ({q_label}): {init_mass_avg:.3} kg")
    
    # Compute effective heat of combustion
    eff_hoc = thr / init_mass_avg
    cone_eff_hoc_all[q_label] = eff_hoc
    
    print(f"HOC_eff ({q_label}): {eff_hoc:.9} kJ/kg\n")

    
# Compute the average effective heat of combustion
hocs = list()
for q_label in cone_thr_all:
    hocs.append(cone_eff_hoc_all[q_label])
    

avg_hoc = np.average(hocs)
print(f"\nHOC_eff (average): {avg_hoc:.10} kJ/kg\n")

